### IMPORTS

In [1]:
import warnings
# Ignore all FutureWarnings
warnings.filterwarnings("ignore", category=FutureWarning)

import re
import math
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn import tree
from sklearn.model_selection import RandomizedSearchCV, train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
from sklearn.svm import SVC
import random
import nltk
nltk.download('punkt')
from nltk.tokenize import sent_tokenize
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import spacy
from spacy.lang.en.stop_words import STOP_WORDS

# Make results reproducible
random.seed(100)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\UFC\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


### Load Data

In [2]:
# Labelled data loading
data = pd.read_csv('A2_customer_churn_labeled.csv')
data.head()

,ID,credit_score,tenure,balance,number_of_products,has_credit_card,is_active_member,salary,customer_profile,Y
0,0,585,4,0.00,2,0,1,101728.46,This customer is a 44-year-old female from Spa...,0
1,1,743,6,140348.56,2,1,1,163254.39,This customer is a 32-year-old female from Ger...,0
2,2,527,10,136733.23,1,1,1,57589.29,This customer is a 41-year-old female from Ger...,0
3,3,732,6,98792.40,1,1,0,81491.70,This customer is a 45-year-old female from Ger...,1
4,4,641,3,0.00,2,1,0,116466.19,This customer is a 38-year-old female from Fra...,0


In [3]:
print('Shape of labeled data: ', data.shape)

Shape of labeled data:  (7000, 10)


### Helper functions for adding new columns 

In [4]:
def get_last_two_sentences(text):
    sentences = sent_tokenize(text)

    # Get the last 2 sentences
    last_two_sentences = sentences[-2:]

    return ' '.join(last_two_sentences)

# Create an object instance sih of SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

# Function that returns compound polarity score of the text
def get_polarity(text):
    # Get the polarity scores of the passed text
    return sia.polarity_scores(text)['compound']

# Load spaCy's English tokenizer and tagger
nlp = spacy.load("en_core_web_sm")

# Define a function to perform tokenization, stopwords removal, and lemmatization
def preprocess_text(text):
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if token.text.lower() not in STOP_WORDS]
    return " ".join(tokens)


### Add new columns

In [5]:
def add_new_columns(dataset):
     #  Just comment the line for the column that is not needed
    
    dataset['age'] = dataset['customer_profile'].apply(lambda x: int(re.findall('(\d+)-year-old', x)[0]))
    dataset['gender'] = dataset['customer_profile'].apply(lambda x: re.findall('(male|female)', x)[0])
    dataset['country'] = dataset['customer_profile'].apply(lambda x: re.findall('from (\w+)', x)[0])
    dataset['last_lines'] = dataset['customer_profile'].apply(lambda text: get_last_two_sentences(text))
    dataset['customer_profile_polarity'] = dataset['customer_profile'].apply(lambda text: get_polarity(text))
    dataset['last_lines_polarity'] = dataset['last_lines'].apply(lambda text: get_polarity(text))
    dataset['customer_profile_tokenized'] = dataset['customer_profile'].apply(lambda text: preprocess_text(text))
    return dataset

In [6]:
data1 = add_new_columns(data)

### Drop any existing columns

In [7]:
def drop_any_existing_columns(dataset, columns = ['ID']):
    dataset = dataset.drop(columns=columns, inplace=False)
    return dataset

In [8]:
columns_to_drop = ['ID', 'customer_profile', 'last_lines', 'customer_profile_tokenized']

data2 = drop_any_existing_columns(data1, columns = columns_to_drop)

### Any required feature transformations

In [9]:
data2.head()

,credit_score,tenure,balance,number_of_products,has_credit_card,is_active_member,salary,Y,age,gender,country,customer_profile_polarity,last_lines_polarity
0,585,4,0.00,2,0,1,101728.46,0,44,female,Spain,0.7311,0.4512
1,743,6,140348.56,2,1,1,163254.39,0,32,female,Germany,0.7845,0.6486
2,527,10,136733.23,1,1,1,57589.29,0,41,female,Germany,0.7845,0.6486
3,732,6,98792.40,1,1,0,81491.70,1,45,female,Germany,0.4482,-0.3089
4,641,3,0.00,2,1,0,116466.19,0,38,female,France,0.4482,-0.3089


In [10]:
### Dummy variablize
data3 = pd.get_dummies(data2, columns = ['gender', 'country'], drop_first=True )
print(data3.columns)
data3.head()

Index(['credit_score', 'tenure', 'balance', 'number_of_products',
       'has_credit_card', 'is_active_member', 'salary', 'Y', 'age',
       'customer_profile_polarity', 'last_lines_polarity', 'gender_male',
       'country_Germany', 'country_Spain'],
      dtype='object')


,credit_score,tenure,balance,number_of_products,has_credit_card,is_active_member,salary,Y,age,customer_profile_polarity,last_lines_polarity,gender_male,country_Germany,country_Spain
0,585,4,0.00,2,0,1,101728.46,0,44,0.7311,0.4512,False,False,True
1,743,6,140348.56,2,1,1,163254.39,0,32,0.7845,0.6486,False,True,False
2,527,10,136733.23,1,1,1,57589.29,0,41,0.7845,0.6486,False,True,False
3,732,6,98792.40,1,1,0,81491.70,1,45,0.4482,-0.3089,False,True,False
4,641,3,0.00,2,1,0,116466.19,0,38,0.4482,-0.3089,False,False,False


## Models implementation

### Model evaluation metrics

In [12]:
def calc_f1_score(model, X=None, y=None, dataset=None, type='macro'):
    if X is None and y is None and dataset is not None:
        y = dataset['Y']
        X = dataset.drop(columns='Y')
    y_pred = model.predict(X)
    return f1_score(y, y_pred, average='macro')

In [13]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt 

In [49]:
data3['gender_male'] = data3['gender_male'].astype(int)
data3['country_Germany'] = data3['country_Germany'].astype(int)
data3['country_Spain'] = data3['country_Spain'].astype(int)
data3.head()

,credit_score,tenure,balance,number_of_products,has_credit_card,is_active_member,salary,Y,age,customer_profile_polarity,last_lines_polarity,gender_male,country_Germany,country_Spain
0,585,4,0.00,2,0,1,101728.46,0,44,0.7311,0.4512,0,0,1
1,743,6,140348.56,2,1,1,163254.39,0,32,0.7845,0.6486,0,1,0
2,527,10,136733.23,1,1,1,57589.29,0,41,0.7845,0.6486,0,1,0
3,732,6,98792.40,1,1,0,81491.70,1,45,0.4482,-0.3089,0,1,0
4,641,3,0.00,2,1,0,116466.19,0,38,0.4482,-0.3089,0,0,0


### Creating tensorflow datasets

In [60]:
# Perform train test split
X_train_full, X_test, y_train_full, y_test = train_test_split(data3.drop(columns='Y'), data3['Y'], test_size=0.1, random_state=42)

In [61]:
X_train_full = np.array(X_train_full, dtype=np.float32)
X_test = np.array(X_test)
y_train_full = np.array(y_train_full, dtype=np.float32)
y_test = np.array(y_test)

In [62]:
print('Shape of X_train is: ', X_train_full.shape)
print('Shape of X_test is: ', X_test.shape)

Shape of X_train is:  (6300, 13)
Shape of X_test is:  (700, 13)


In [63]:
# Shuffle X_train_full and y_train_full
shuffled_indices = np.random.permutation(X_train_full.shape[0])
X_train_full = X_train_full[shuffled_indices]
y_train_full = y_train_full[shuffled_indices]

print(X_train_full.shape, y_train_full.shape)
print(X_test.shape, y_test.shape)

(6300, 13) (6300,)
(700, 13) (700,)


In [64]:
import math
N = X_train_full.shape[0]
i = math.floor(0.9*N)

# Splitting the full training data X_train into two parts (10% for validation and 90% for training)
X_train, y_train = X_train_full[:i], y_train_full[:i]
X_valid, y_valid = X_train_full[i:], y_train_full[i:]

In [65]:
print('Training set', X_train.shape, y_train.shape)
print('Validation set', X_valid.shape, y_valid.shape)
print('Test set', X_test.shape, y_test.shape)

Training set (5670, 13) (5670,)
Validation set (630, 13) (630,)
Test set (700, 13) (700,)


In [66]:
num_classes = np.unique(y_train_full).shape[0]
num_classes

2

## Deep Neural Network class

In [67]:

from tensorflow import keras
# Keras tensorflow libraries and packages
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential

In [75]:
class F1Score(tf.keras.metrics.Metric):
    def __init__(self, name='f1_score', **kwargs):
        super(F1Score, self).__init__(name=name, **kwargs)
        self.true_positives = self.add_weight(name='true_positives', initializer='zeros')
        self.false_positives = self.add_weight(name='false_positives', initializer='zeros')
        self.false_negatives = self.add_weight(name='false_negatives', initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.cast(y_true, tf.bool)
        y_pred = tf.cast(y_pred, tf.bool)

        true_positives = tf.reduce_sum(tf.cast(tf.logical_and(y_true, y_pred), tf.float32))
        false_positives = tf.reduce_sum(tf.cast(tf.logical_and(tf.logical_not(y_true), y_pred), tf.float32))
        false_negatives = tf.reduce_sum(tf.cast(tf.logical_and(y_true, tf.logical_not(y_pred)), tf.float32))

        self.true_positives.assign_add(true_positives)
        self.false_positives.assign_add(false_positives)
        self.false_negatives.assign_add(false_negatives)

    def result(self):
        precision = self.true_positives / (self.true_positives + self.false_positives + tf.keras.backend.epsilon())
        recall = self.true_positives / (self.true_positives + self.false_negatives + tf.keras.backend.epsilon())
        f1 = 2 * (precision * recall) / (precision + recall + tf.keras.backend.epsilon())
        return f1

In [76]:
# A DNN class (or Deep Neural Network class) to facilitate the building training and testing of a deep learning model
class DNN:
    def __init__(self, input_features=13, n1=10, n2=5, act='relu', num_classes=2):
        # Initialize the model structure, and number of classes
        self.input_features = input_features
        self.n1 = n1
        self.n2 = n2
        self.act = act
        self.num_classes = num_classes
        self.model = Sequential()
        
    def build_model(self, show_summary=True):
        # Building model with n1 neurons in first hidden layer and n2 neurons in second hidden layer
        self.model = Sequential()
        self.model.add(Dense(units=self.n1, activation=self.act, name='hidden_layer_1', input_shape=(self.input_features,)))
        self.model.add(Dense(units=self.n2, activation=self.act, name='hidden_layer_2'))
        self.model.add(Dense(units=self.num_classes, activation='softmax', name='output_layer'))
        self.model.build()
        if show_summary:
            print('Following is the summary of the built model: ')
            self.model.summary()

    def compile_model(self, optimizer=keras.optimizers.Adam(learning_rate=0.001)):
        # Compiling the model with a particular configurable optimizer
        self.model.compile(optimizer=optimizer,
                           loss='sparse_categorical_crossentropy',
                           metrics=['accuracy', F1Score()])

    def train_model(self, X_train, y_train, X_valid, y_valid, batch_size=64, epochs=50, callbacks=[], verbose=1):
        # Train the model
        history = self.model.fit(x=X_train, 
                                 y=y_train, 
                                 batch_size=batch_size,
                                 epochs=epochs,
                                 validation_data=(X_valid, y_valid), 
                                 callbacks=callbacks,
                                 verbose=verbose)
        return history

    def evaluate_model(self, X_test, y_test, verbose=1):
        return self.model.evaluate(X_test, y_test, verbose=verbose)
                    

In [80]:
from itertools import product

# Choices for n1 = number of neurons in hidden layer 1
n1_choices = [5, 8, 10]
# Choices for n2 = number of neurons in hidden layer 2
n2_choices = [5, 8, 10]
# Choices for activation function
act_choices = ['sigmoid', 'tanh', 'relu']

# Learning Rate chosen
learning_rate = 0.001

# Grid containing combinations of choices for n1, n2 and act
grid = list(product(n1_choices, n2_choices, act_choices))

best_accuracy=0
best_hyperparameters = None
best_model = None

# Tuning the hyperparameters
for n1, n2, act in grid:
    print('\n\nModel with n1={}, n2={} and {} activation function =>'.format(n1, n2, act))
    dnn = DNN(input_features=X_train.shape[1], n1=n1, n2=n2, act=act, num_classes=num_classes)
    dnn.build_model(show_summary=False)

    # Optimizer chosen
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

    dnn.compile_model(optimizer=optimizer)
    print('\tTraining the model...')
    dnn.train_model(X_train, y_train, X_valid, y_valid, batch_size=64, epochs=50, verbose=0)
    model_loss, model_accuracy, f1_score = dnn.evaluate_model(X_test, y_test, verbose=0)
    print('\tTrained successfully')
    print('\tLoss on test dataset: ', model_loss)
    print('\tAccuracy on test dataset: ', model_accuracy)
    print('\tF1 Score on test dataset: ', f1_score)
    if f1_score > best_accuracy:
        best_accuracy = f1_score
        best_hyperparameters = (n1, n2, act)
        best_model = dnn.model
#         best_model.save('models/bestDNN_Part2_Q4.keras')

print('After tuning of the hyperparamters using grid search, \
the best hyperparameters are found to be: n1={}, n2={} and act={} with an accuracy of {} on the test dataset'.format(best_hyperparameters[0],
                                                                                                                     best_hyperparameters[1],
                                                                                                                     best_hyperparameters[2], 
                                                                                                                     best_accuracy))
                                                                         



Model with n1=5, n2=5 and sigmoid activation function =>
	Training the model...
	Trained successfully
	Loss on test dataset:  0.499572217464447
	Accuracy on test dataset:  0.7985714077949524
	F1 Score on test dataset:  0.335315078496933


Model with n1=5, n2=5 and tanh activation function =>
	Training the model...
	Trained successfully
	Loss on test dataset:  0.49598437547683716
	Accuracy on test dataset:  0.7985714077949524
	F1 Score on test dataset:  0.335315078496933


Model with n1=5, n2=5 and relu activation function =>
	Training the model...
	Trained successfully
	Loss on test dataset:  2.439152717590332
	Accuracy on test dataset:  0.7985714077949524
	F1 Score on test dataset:  0.335315078496933


Model with n1=5, n2=8 and sigmoid activation function =>
	Training the model...
	Trained successfully
	Loss on test dataset:  0.4953884482383728
	Accuracy on test dataset:  0.7985714077949524
	F1 Score on test dataset:  0.335315078496933


Model with n1=5, n2=8 and tanh activation fun

In [82]:
from sklearn.neural_network import MLPClassifier

In [ ]:
mlp_classifier = MLPClassifier()

In [78]:
dnn.evaluate_model(X_test, y_test, verbose=0)

[0.49697449803352356, 0.7985714077949524, 0.335315078496933]

In [ ]:
boost_classifier.fit(data3.drop(columns='Y'), data3['Y'])

## Sampling Smote

In [24]:
from imblearn.over_sampling import SMOTE
import pandas as pd

def sampling_methods(X,sampling_method,sampling_ratio = None):
    
    minority_class_label = 1
    
    if sampling_method == 'over':

        minority_samples = X[X['Y'] == minority_class_label]
        majority_samples = X[X['Y'] != minority_class_label]
        
        if sampling_ratio == None:
            ratio = len(majority_samples) // len(minority_samples)
        else:
            ratio = sampling_ratio
        oversampled_minority = minority_samples.sample(n=len(minority_samples) * (ratio), replace=True)

        df = pd.concat([majority_samples, oversampled_minority], ignore_index=True)
    
    elif sampling_method == 'under':

        minority_samples = X[X['Y'] == minority_class_label]
        majority_samples = X[X['Y'] != minority_class_label]

        if sampling_ratio == None:
            ratio = len(minority_samples) / len(majority_samples)
        else:
            ratio = sampling_ratio
        
        undersampled_majority = majority_samples.sample(frac=ratio, random_state= 42)

        df = pd.concat([minority_samples, undersampled_majority], ignore_index=True)
    
    elif sampling_method == 'SMOTE':
        
        minority_samples = X[X['Y'] == minority_class_label]
        majority_samples = X[X['Y'] != minority_class_label]

        if sampling_ratio == None:
            ratio = 'auto'
        else:
            ratio = sampling_ratio
        

        features = X.drop(columns=['Y'])
        label = X['Y']
        
        smote = SMOTE(sampling_strategy= ratio, random_state=42)
        X_resampled, y_resampled = smote.fit_resample(features, label)
        
        df = pd.concat([pd.DataFrame(X_resampled, columns=features.columns),
                                  pd.Series(y_resampled, name='Y')], axis=1)

    return df

In [25]:
data3_smote = sampling_methods(data3, 'SMOTE')

In [26]:
data3_smote['Y'].value_counts()

Y
0    5565
1    5565
Name: count, dtype: int64

In [33]:
# Perform train test split
X_train, X_test, y_train, y_test = train_test_split(data3_smote.drop(columns='Y'), data3_smote['Y'], test_size=0.2, random_state=42)

### Model 2 Gradient Boosting with hyperparameter tuning and sampling methods

In [ ]:
# Generate values for lambdas
pows = np.arange(-10, -0.1, 0.1)
lambdas = 10**pows
length_lambdas = len(lambdas)

# Initialize arrays to store MSE values
train_scores = []
test_scores = []

for lambd in lambdas:
    print(lambd)
    # 1. Fit the boosting model with different lambda
    boost_classifier_hyp = GradientBoostingClassifier(n_estimators=1000, random_state=42, learning_rate=lambd)

    # Fit the model to the training data
    boost_classifier_hyp.fit(X_train, y_train)
    
    # 4. Calculate and save the f1 score
    train_score = calc_f1_score(boost_classifier_hyp, X_train, y_train)
    test_score = calc_f1_score(boost_classifier_hyp, X_test, y_test)
    
    train_scores.append(train_score)
    test_scores.append(test_score)

1e-10
1.2589254117941662e-10
1.584893192461111e-10
1.9952623149688748e-10


In [63]:
import matplotlib.pyplot as plt

# Set up subplots with 1 row and 2 columns
fig, axs = plt.subplots(1, 2, figsize=(12, 5))

# Plot the Training MSE
axs[0].plot(lambdas, train_scores, 'bo-')
axs[0].set_xlabel("Shrinkage values lambda")
axs[0].set_ylabel("Training F1 Scores")
axs[0].set_title("F1 Score on Training data")

# Plot the Testing MSE
axs[1].plot(lambdas, test_scores, 'bo-')
axs[1].set_xlabel("Shrinkage values lambda")
axs[1].set_ylabel("Testing F1 Scores")
axs[1].set_title("F1 Score on Testing data")

# Adjust spacing between subplots
plt.tight_layout()

# Show the plots
plt.show()

(1000, 5)

In [ ]:
best_lambda = ?

In [ ]:
boost_classifier = GradientBoostingClassifier(n_estimators=1000, random_state=42, learning_rate=best_lambda)

# Fit the model to the training data
boost_classifier.fit(data3_smote.drop(columns='Y'), data3_smote['Y'])


### Generating the Kaggle Submission File

In [17]:
final_model = boost_classifier

In [18]:
X_kaggle_test = pd.read_csv('A2_customer_churn_kaggle.csv')

In [19]:
# Add columns
X_kaggle_test1 = add_new_columns(X_kaggle_test)

In [20]:
# Drop columns
columns_to_drop = ['ID', 'Y', 'customer_profile', 'last_lines', 'customer_profile_tokenized']
X_kaggle_test1 = drop_any_existing_columns(X_kaggle_test1, columns = columns_to_drop)

In [21]:
### Dummy variablize
X_kaggle_test1 = pd.get_dummies(X_kaggle_test1, columns = ['gender', 'country'], drop_first=True )

In [22]:
print('Shape of kaggle test: ', X_kaggle_test1.shape)
X_kaggle_test1.head()

Shape of kaggle test:  (2010, 13)


,credit_score,tenure,balance,number_of_products,has_credit_card,is_active_member,salary,age,customer_profile_polarity,last_lines_polarity,gender_male,country_Germany,country_Spain
0,652,4,59486.31,1,1,0,163944.19,48,0.4482,-0.3089,True,False,False
1,714,4,0.00,2,1,1,37605.90,29,0.7845,0.6486,True,False,True
2,733,3,100337.96,3,1,0,48559.19,34,0.4482,-0.3089,True,True,False
3,577,8,79757.21,1,1,0,135650.72,43,0.4482,-0.3089,True,False,True
4,600,2,119755.00,1,1,1,21852.91,30,0.7845,0.6486,True,True,False


In [23]:
y_pred = final_model.predict(X_kaggle_test1)
df_pred = pd.concat([X_kaggle_test['ID'], pd.DataFrame(y_pred,columns=['Y'])], axis = 1)
df_pred.to_csv('kaggle_pred_values.csv',index=False)